# 07 — Slowly Changing Dimensions (SCD) with Delta Lake

This notebook demonstrates how to implement **Slowly Changing Dimensions (SCD)** Type 1 and Type 2 using Delta Lake's `MERGE INTO`.

- **SCD Type 1:** Overwrite the existing record with new data. Keeps no history of previous values.
- **SCD Type 2:** Insert a new record with the new data, and mark the old record as expired. Maintains full history of changes.

In [ ]:
from spark_session import get_spark
from pyspark.sql import functions as F

spark = get_spark("07-scd-delta")
TABLE_T1 = "s3a://spark-warehouse/delta/scd/type1"
TABLE_T2 = "s3a://spark-warehouse/delta/scd/type2"

## SCD Type 1 (Overwrite)

Let's create an initial `customers` table.

In [ ]:
customers_t1 = spark.createDataFrame([
    (1, "Alice", "New York"),
    (2, "Bob", "London"),
    (3, "Charlie", "Paris")
], ["customer_id", "name", "city"])

customers_t1.write.format("delta").mode("overwrite").save(TABLE_T1)
spark.read.format("delta").load(TABLE_T1).orderBy("customer_id").show()

Now, let's process some incoming updates. Alice moves to 'San Francisco', and we get a new customer 'David'.

In [ ]:
updates_t1 = spark.createDataFrame([
    (1, "Alice", "San Francisco"), # Update
    (4, "David", "Tokyo")          # Insert
], ["customer_id", "name", "city"])

updates_t1.createOrReplaceTempView("updates_t1")
spark.sql(f"CREATE OR REPLACE TEMP VIEW customers_t1 AS SELECT * FROM delta.`{TABLE_T1}`")

spark.sql(f"""
MERGE INTO delta.`{TABLE_T1}` t
USING updates_t1 u
ON t.customer_id = u.customer_id
WHEN MATCHED THEN 
  UPDATE SET t.city = u.city
WHEN NOT MATCHED THEN 
  INSERT *
""")

spark.read.format("delta").load(TABLE_T1).orderBy("customer_id").show()

## SCD Type 2 (Historical Versioning)

With Type 2, we track historical changes using `is_current`, `start_date`, and `end_date` columns.

In [ ]:
customers_t2 = spark.createDataFrame([
    (1, "Alice", "New York", True, "2023-01-01", None),
    (2, "Bob", "London", True, "2023-01-01", None),
    (3, "Charlie", "Paris", True, "2023-01-01", None)
], ["customer_id", "name", "city", "is_current", "start_date", "end_date"])

customers_t2.write.format("delta").mode("overwrite").save(TABLE_T2)
spark.read.format("delta").load(TABLE_T2).orderBy("customer_id").show()

To do SCD Type 2 with Delta Lake efficiently, we typically use a clever two-part MERGE pattern:
we union the incoming updates into two sets:
1. Rows that will update existing records (to mark them as `is_current=false`). We map these using the `customer_id` as the merge key.
2. Rows that will be inserted as new records (both new customers, and new versions of updated customers). We use a `NULL` merge key for the updates, forcing them into the `NOT MATCHED` clause to perform an insert.

In [ ]:
updates_t2 = spark.createDataFrame([
    (1, "Alice", "San Francisco", "2023-06-01"), # Update
    (4, "David", "Tokyo", "2023-06-01")          # Insert
], ["customer_id", "name", "city", "update_date"])

updates_t2.createOrReplaceTempView("updates_t2")
spark.sql(f"CREATE OR REPLACE TEMP VIEW customers_t2 AS SELECT * FROM delta.`{TABLE_T2}`")

spark.sql(f"""
MERGE INTO delta.`{TABLE_T2}` t
USING (
    -- These rows will either UPDATE the current records of existing customers or INSERT new customers
    SELECT updates_t2.customer_id as merge_key, updates_t2.*
    FROM updates_t2
    
    UNION ALL
    
    -- These rows will INSERT new current records for existing customers whose city has changed
    SELECT NULL as merge_key, updates_t2.*
    FROM updates_t2 JOIN customers_t2 
    ON updates_t2.customer_id = customers_t2.customer_id 
    WHERE customers_t2.is_current = true AND customers_t2.city <> updates_t2.city
) staged_updates
ON t.customer_id = staged_updates.merge_key

WHEN MATCHED AND t.is_current = true AND t.city <> staged_updates.city THEN
    -- Mark the existing record as no longer current
    UPDATE SET is_current = false, end_date = staged_updates.update_date

WHEN NOT MATCHED THEN
    -- Insert the new record (either a brand new customer or a new version of an updated customer)
    INSERT (customer_id, name, city, is_current, start_date, end_date)
    VALUES (staged_updates.customer_id, staged_updates.name, staged_updates.city, true, staged_updates.update_date, null)
""")

spark.read.format("delta").load(TABLE_T2).orderBy("customer_id", "start_date").show()


In [ ]:
# Release the SparkSession
spark.stop()